# Multi-Agent CMG

This tutorial introduces MASA's multi-agent constrained Markov game path with a tiny repeated Chicken matrix game. The goal is to see how PettingZoo parallel environments return per-agent dictionaries, how `make_marl_env` adds MASA labels and CMG budgets, and how per-agent costs can feed a shared safety constraint.

The matching static docs page is at `docs/Tutorials/Multi-Agent/Multi-Agent CMG.md`.

## CPU-first setup

Keep the notebook portable and quiet before importing MASA modules.

In [ ]:
import os

os.environ.setdefault("JAX_PLATFORMS", "cpu")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

## Imports

`ChickenMatrix` is a PettingZoo `ParallelEnv`: observations, rewards, termination flags, truncation flags, and infos are dictionaries keyed by agent id.

In [ ]:
from pettingzoo import ParallelEnv

from masa.common.constraints.multi_agent.cmg import Budget
from masa.common.utils import make_marl_env
from masa.envs.multiagent.matrix.chicken import Actions, ChickenMatrix, cost_fn, label_fn

## Raw PettingZoo parallel environment

Start without MASA wrappers. The raw environment shows the core PettingZoo shape: each agent has its own observation and action space, and `step` consumes an action dictionary.

In [ ]:
raw_env = ChickenMatrix(max_moves=2)

assert isinstance(raw_env, ParallelEnv)
print("possible_agents:", raw_env.possible_agents)
for agent in raw_env.possible_agents:
    print(agent, "observation_space=", raw_env.observation_space(agent), "action_space=", raw_env.action_space(agent))

obs, infos = raw_env.reset(seed=0)
print("reset obs keys:", list(obs))
print("reset info keys:", list(infos))

actions = {"player_0": Actions.Swerve, "player_1": Actions.Straight}
obs, rewards, terminations, truncations, infos = raw_env.step(actions)
print("rewards:", rewards)
print("terminations:", terminations)
print("truncations:", truncations)
print("infos:", infos)

assert set(obs) == {"player_0", "player_1"}
assert set(rewards) == {"player_0", "player_1"}
raw_env.close()

## Chicken safety semantics

In Chicken, both agents choosing `Straight` causes a crash. The environment's `label_fn` emits `crash` and `unsafe`, and the default `cost_fn` maps that unsafe label set to cost `1.0`.

In [ ]:
crash_obs = ChickenMatrix(max_moves=1)._obs()
crash_obs[1] = 1  # player_0_straight
crash_obs[3] = 1  # player_1_straight
crash_obs[4] = 1  # crash flag

labels = label_fn(crash_obs)
print("Straight/Straight labels:", sorted(labels))
print("cost:", cost_fn(labels))

assert {"crash", "unsafe"}.issubset(labels)
assert cost_fn(labels) == 1.0

## Build a MASA CMG environment

`make_marl_env` builds the standard multi-agent wrapper path: raw PettingZoo env, then `LabelledParallelEnv`, then `ConstrainedMarkovGameEnv`. The budgets below include two individual budgets and one shared budget over both agents.

In [ ]:
AGENTS = ("player_0", "player_1")


def build_chicken_cmg(max_moves=3):
    return make_marl_env(
        "chicken_matrix",
        "cmg",
        env_kwargs={"max_moves": max_moves},
        budgets=[
            Budget(amount=1.0, agents=("player_0",), name="player_0_budget"),
            Budget(amount=1.0, agents=("player_1",), name="player_1_budget"),
            Budget(amount=1.5, agents=("player_0", "player_1"), name="shared"),
        ],
    )


env = build_chicken_cmg(max_moves=3)
obs, infos = env.reset(seed=0)
print("constraint type:", env.constraint_type)
print("reset labels:", {agent: infos[agent]["labels"] for agent in AGENTS})
print("reset metrics:", env.constraint_step_metrics())

assert env.constraint_type == "cmg"
assert all(infos[agent]["labels"] == set() for agent in AGENTS)
assert env.constraint_step_metrics()["satisfied"] == 1.0
env.close()

## Rollout helper

The helper below keeps the printed rows compact while preserving the original `infos[agent]["labels"]`, step metrics, and episode metrics for assertions.

In [ ]:
def action_name(action):
    return Actions(int(action)).name


def summarize_actions(actions):
    return {agent: action_name(actions[agent]) for agent in AGENTS}


def run_cmg_script(script, *, seed=0, max_moves=3):
    env = build_chicken_cmg(max_moves=max_moves)
    obs, infos = env.reset(seed=seed)
    rows = []

    for step, actions in enumerate(script, start=1):
        obs, rewards, terminations, truncations, infos = env.step(actions)
        metrics = env.constraint_step_metrics()
        episode_metrics = env.constraint_episode_metrics()
        labels_by_agent = {agent: sorted(infos[agent]["labels"]) for agent in AGENTS}

        rows.append(
            {
                "step": step,
                "actions": summarize_actions(actions),
                "rewards": rewards,
                "labels": labels_by_agent,
                "truncations": truncations,
                "metrics": metrics,
                "episode_metrics": episode_metrics,
                "compact": {
                    "step": step,
                    "actions": summarize_actions(actions),
                    "rewards": rewards,
                    "player_0_cost": metrics["player_0_cost"],
                    "player_1_cost": metrics["player_1_cost"],
                    "shared_cum_cost": metrics["shared_cum_cost"],
                    "shared_satisfied": metrics["shared_satisfied"],
                    "satisfied": metrics["satisfied"],
                },
            }
        )

    env.close()
    return rows


def compact_table(rows):
    return [row["compact"] for row in rows]

## Safe rollout

`Swerve/Swerve` and `Straight/Swerve` produce rewards without crash labels. Both agents have zero cost, so every budget remains satisfied.

In [ ]:
safe_script = [
    {"player_0": Actions.Swerve, "player_1": Actions.Swerve},
    {"player_0": Actions.Straight, "player_1": Actions.Swerve},
]

safe_rows = run_cmg_script(safe_script, max_moves=3)
compact_table(safe_rows)

In [ ]:
safe_final = safe_rows[-1]
safe_metrics = safe_final["metrics"]

assert safe_metrics["player_0_cost"] == 0.0
assert safe_metrics["player_1_cost"] == 0.0
assert safe_metrics["shared_cum_cost"] == 0.0
assert safe_metrics["shared_satisfied"] == 1.0
assert safe_metrics["satisfied"] == 1.0
assert "unsafe" not in safe_final["labels"]["player_0"]
assert "unsafe" not in safe_final["labels"]["player_1"]

## Unsafe shared-budget rollout

`Straight/Straight` gives both agents the unsafe crash label. Each individual budget has amount `1.0`, so each agent can still be individually within budget. The shared budget has amount `1.5`, so the combined cost `2.0` fails the shared constraint.

In [ ]:
unsafe_script = [
    {"player_0": Actions.Straight, "player_1": Actions.Straight},
]

unsafe_rows = run_cmg_script(unsafe_script, max_moves=1)
compact_table(unsafe_rows)

In [ ]:
unsafe_final = unsafe_rows[-1]
unsafe_metrics = unsafe_final["metrics"]
unsafe_episode_metrics = unsafe_final["episode_metrics"]

assert "unsafe" in unsafe_final["labels"]["player_0"]
assert "unsafe" in unsafe_final["labels"]["player_1"]
assert unsafe_metrics["player_0_cost"] == 1.0
assert unsafe_metrics["player_1_cost"] == 1.0
assert unsafe_metrics["player_0_budget_satisfied"] == 1.0
assert unsafe_metrics["player_1_budget_satisfied"] == 1.0
assert unsafe_metrics["shared_cost"] == 2.0
assert unsafe_metrics["shared_cum_cost"] == 2.0
assert unsafe_metrics["shared_satisfied"] == 0.0
assert unsafe_metrics["satisfied"] == 0.0
assert unsafe_final["truncations"]["player_0"] is True
assert unsafe_final["truncations"]["player_1"] is True
assert unsafe_episode_metrics["shared_cum_cost"] == 2.0
assert unsafe_episode_metrics["shared_satisfied"] == 0.0
assert unsafe_episode_metrics["satisfied"] == 0.0

## Wrap-up

The PettingZoo parallel API keeps each agent's data in dictionaries. MASA adds labels per agent, converts those labels to per-agent costs, and lets CMG budgets aggregate over any subset of agents. That is why the final rollout can be individually acceptable for both agents while still failing the shared safety budget.